# Klassificering av ansiktsuttryck med CNN

In [ ]:
# Importerar bibliotek för hela min kod 
import time, os 
import numpy as np,pandas as pd, matplotlib.pyplot as plt 


from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix,ConfusionMatrixDisplay,classification_report

import tensorflow as tf
from tensorflow import keras 
from tensorflow.keras import layers, models

from tensorflow.keras.utils import load_img, img_to_array

# slump seed  
tf.keras.utils.set_random_seed(42)

# Laddar dataset 

- jag använder  image_dataset_from_directory för att läsa in bilderna i gråskala och ändrar storleken till 48x48 pixlar. 
- jag upp träningsmappen med validation_split: 80% av bilderna tränar modellen på, och 20% sparas som validering för att testa modellen under träningens gång.

In [ ]:
# Laddar in data från mappar i datan 
train_path = "FER-2013/train"
test_path = "FER-2013/test"
 

img_size = (48, 48)  # Ändrar alla bilder till storlek 48 * 48 
batch_size = 32   # hur många bilder modellen läser in samtidigt 

# Lägger till validation_split för att övervaka overfittning 
train_data =tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset= "training",
    seed = 42,
    color_mode ="grayscale",
    image_size =img_size,
    batch_size = batch_size,
    label_mode = "int" 
)


val_data =tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split= 0.2,
    subset= "validation",
    seed =42,
    color_mode="grayscale",
    image_size =img_size,
    batch_size =batch_size,
    label_mode="int"
)



test_data =tf.keras.utils.image_dataset_from_directory(
    test_path,
    color_mode ="grayscale",
    image_size =img_size,
    batch_size= batch_size
)

# hämtar klassernas name 
class_names =train_data.class_names

# skriver ut klasserna 
print(class_names)

# Undersöka datan

Här kollar jag vilka klasser som finns och använder en enkel loop för att räkna antalet bilder i varje mapp. Det gör jag för att kontrollera klassbalansen, så jag vet om vissa känslor har mycket fler bilder än andra.

In [ ]:
# Visar vilka klasser som finns 

print("klasser:")
print(class_names)


# Räknar antal bild i varje klass 

for category in class_names:
    path =os.path.join(train_path, category)   # sökväg till klassen 
    num_images =len(os.listdir(path))       # Räknar bilder 

    print(f"{category}:{num_images} bilder ")


    

# Visar exemple bilder 

Jag plockar ut en grupp bilder från datasetet och visar 18 exempel på skärmen i gråskala med rätt känsla som rubrik, så att jag ser hur datan ser ut.

In [ ]:
plt.figure(figsize=(10, 6))
plt.suptitle("Example bilder från dataset", fontsize=15,fontweight="bold", color="navy")
for images,labells in train_data.take(1):
    for i in range(18):
        plt.subplot(3, 6, i + 1)

        plt.imshow(images[i].numpy().squeeze(), cmap= "gray")
        plt.title(class_names[labells[i]])
        plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# disgust hade väldigt få bilder så jag lade till augmentation

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),     
    layers.RandomRotation(0.1),           
    layers.RandomZoom(0.1),                 #testade 0.2 men blev konstig
])

# Bygger och kompilerar  CNN -model

In [ ]:
# Enkel modell utan dropout
enkla_model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(48, 48, 1)), #inbyggd normalisering

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2 ,2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64,  activation="relu"),
    layers.Dense(len(class_names),  activation ="softmax")
])

enkla_model.compile(
    optimizer="Adam",
    loss ="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# # Bättre modell med dropout och augmentation  

bättre_model = models.Sequential([
    data_augmentation,
    layers.Rescaling(1./255, input_shape=(48, 48, 1)),



    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(256, activation= "relu"),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation="softmax")
])


bättre_model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=0.001),
    loss= "sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

bättre_model.build(input_shape=(None, 48, 48, 1))
bättre_model.summary()

# Tränar modellen 


In [ ]:
# Early stopping ,  den slutar träna om det inte förbättras
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

EPOCHS = 30

print("Tränar enkla modellen")
start_b = time.time()
träning_enkel = enkla_model.fit(
    train_data,
    validation_data=val_data,
    epochs=8 
)
print(f"Träningstid enkla_model: {time.time() - start_b:.2f} sekunder\n")

print("tränar bättre_model")
start_i = time.time()
träning_bättre = bättre_model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=[early_stop]
)
print(f"Träningstid bättre model : {time.time() - start_i:.2f} sekunder")

#  Visar accuracy och loss grafer 

In [ ]:
def visa_kurvor(history, title):
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs_range = range(len(acc))

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="Tränings-accuracy", color="crimson", linewidth=2)
    plt.plot(epochs_range, val_acc, label="Validerings-accuracy", color="dodgerblue", linewidth=2)
    plt.legend(loc="lower right")
    plt.title(f"{title} - Accuracy", fontweight="bold", fontsize=12)
    plt.grid(True, alpha=0.2)

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="Tränings-loss", color="crimson", linewidth=2)
    plt.plot(epochs_range, val_loss, label="Validerings-loss", color="dodgerblue", linewidth=2)
    plt.legend(loc="upper right")
    plt.title(f"{title} - Loss", fontweight="bold", fontsize=12)
    plt.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

# jämför kurvorna
visa_kurvor(träning_enkel, "enkla model")
visa_kurvor(träning_bättre, "bättre model")

# Utvärdera modellen med test data 



In [ ]:
print("Testar på osedd data:")
test_loss_b, test_acc_b = enkla_model.evaluate(test_data, verbose=0)
print(f"enkla model: {test_acc_b*100:.2f}%")

test_loss_i, test_acc_i = bättre_model.evaluate(test_data, verbose=0)
print(f"bättre model: {test_acc_i*100:.2f}%")



**enkla: 49.57%  bättre: 51.80% — bättre modellen vann men inte med mycket**

# Göra prediktion på ny bild oc visar resultat 

In [ ]:
# här plockar ut en batch från test_data (bilder modellen aldrig har sett)
for test_images, test_labels in test_data.take(1):
    predictions = bättre_model.predict(test_images, verbose=0)
    
    plt.figure(figsize=(10, 6))
    for i in range(6):
        plt.subplot(2, 3, i + 1)
        plt.imshow(test_images[i].numpy().squeeze(), cmap="gray")
        
        pred_label = np.argmax(predictions[i])
        true_label = test_labels[i].numpy()
        confidence = predictions[i][pred_label] * 100
        
        # Grön text om det är rätt, röd om det är fel
        title_color = "green" if pred_label == true_label else 'red'
        
        plt.title(f"Pred: {class_names[pred_label]} ({confidence:.1f}%)\nTrue: {class_names[true_label]}", color=title_color)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Analysera modellen 

In [ ]:
# Samlar in sanna värden och prediktioner för hela testsetet
y_true = []
y_pred = []

for images, labels in test_data:
    preds = bättre_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

# Skapar och visar förvirringsmatrisen
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("Förvirringsmatris - bättre model")
plt.show()

# Detaljerad rapport med precision, recall och f1-score per klass
print("\nKlassificerings rapport")
print(classification_report(y_true, y_pred, target_names=class_names))

## Vad jag testade för att förbättra modellen

Jag provade flera saker under arbetets gång:

- Körde först 20 epochs men bättre modellen fick bara 17% — väldigt dåligt
- Lade till ett extra Conv2D lager (128 filter) men det hjälpte inte
- Tog bort augmentation och testade — blev lite bättre men fortfarande lågt
- Ökade till 40 epochs och fick 51% — mycket bättre
- Slutligen körde jag 30 epochs med augmentation tillbaka och landade på 51.80%

Det som hjälpte mest var att öka antalet epochs. Augmentation gör träningen 
svårare så modellen behöver mer tid på sig att lära sig.

## När är deep learning ett bra val?

Deep learning passar bra för det här problemet eftersom vi jobbar med bilder 
och CNN-modeller är designade för att hitta mönster i pixeldata automatiskt. 
Det hade varit svårt att skriva regler för hand som säger vad som skiljer 
"happy" från "sad" i ett ansikte.

Men deep learning är inte alltid rätt val. Det krävs mycket data — och i det 
här datasetet ser jag  tydligt att klasser med få bilder (t.ex. disgust) är svåra 
för modellen att lära sig. En enklare modell som logistisk regression hade 
förmodligen fungerat dåligt här eftersom bilddata är komplex och högdimensionell.

Deep learning passar när:
- datan är bilder, ljud eller text
- det finns tillräckligt med träningsdata
- mönstren är för komplexa för handskrivna regler

Deep learning passar sämre när:
- datasetet är litet
- man behöver kunna förklara exakt varför modellen fattar ett beslut
- en enklare modell ger liknande resultat med mindre beräkningskraft

# Analysera modellen och diskutera overfitting/underfitting

Hur bra fungerar modellen?   Min bättre modell fick 51.80% och enkla 
modellen 49.57% på testdata. Skillnaden är liten men bättre modellen 
hanterar overfitting bättre vilket syns i graferna.

Tecken på overfitting: I enkla modellen ser jag  tydlig overfitting 
då tränings-loss fortsätter minska medan validerings-loss planar ut,
I bättre modellen har överanpassningen minskat med hjälp av 
Dropout och EarlyStopping, kurvorna följer varandra bättre.


Vad påverkar resultatet mest? Klassbalansen (som undersöktes i steg 3) påverkar mycket. Klasser med väldigt få bilder (t.ex. disgust) har modellen svårare att lära sig, vilket syns i min Confusion Matrix där den ofta felklassificeras som exempelvis angry eller sad.

Modellens begränsningar: Bilderna är lågupplösta (48x48) och i gråskala, vilket gör att subtila nyanser i ansiktsuttryck kan gå förlorade. Modellen är också känslig för obalanserad data.

# Reflektion och slutsats

Vad var svårast i uppgiften? 
Att förstå varför modellen slutade bli bättre 
även om jag lade till fler lager. Tog ett tag innan jag fattade 
att det handlade om dropout och inte antal lager.


Vad lärde du dig?   Att confusion matrix är mycket mer användbar 
än bara accuracy — man ser direkt vilka känslor modellen 
blandar ihop, typ fear och sad.


Vad hade du gjort annorlunda om du började om?  Jag implementerade Data Augmentation för de mindre klasserna.
Jag hade också velat testa fler epochs på enkla modellen för
att se exakt när den börjar överanpassa.


Vilket betyg tycker du att din inlämning motsvarar? VG.

Motivering: Jag har uppfyllt alla grundläggande G krav genom att bygga en fungerande CNN- modell, utvärdera den korrekt och göra prediktioner. Jag har dessutom uppfyllt VG kraven genom att utveckla och systematiskt jämföra två olika modellvarianter (enkla modell  vs bättre model med dropout), tolkat träningskurvorna kritiskt, samt utfört en djupgående felanalys med förvirringsmatris och klassificeringsrapport där datamängdens begränsningar diskuteras.

Jag har även lagt till Data Augmentation för att förbättra 
modellens förmåga att generalisera.